# Northstar FP&A — Financial Planning & Scenario Modelling

**Portfolio project · Walmart Recruiting – Store Sales Forecasting**

This notebook documents the analysis that feeds the Northstar FP&A Vite dashboard. It uses historical weekly department sales across multiple stores, store metadata, holiday flags, promotional markdowns and regional economic features.

## Business questions

1. What does the recent sales trajectory look like across the store network?
2. How do store formats and departments contribute to revenue?
3. How different are holiday and non-holiday sales weeks?
4. Can a time-aware model forecast the next available test horizon without using future actual sales?
5. How do demand, price/mix, contribution margin and operating expense assumptions change the planning outcome?

**Important financial boundary:** the dataset contains `Weekly_Sales`, not COGS, payroll, approved budget or cash flow. The plan is a transparent seasonal reference plan. Contribution and operating contribution are scenario outputs based on editable assumptions, not reported company profit or EBITDA.

Run from the repository root. Download the four competition files into `data/raw/` first. See `START_HERE.md` for exact file names.

## 1. Environment and project imports

The notebook calls the same functions as `run_analysis.py`; it is not a separate analysis implementation.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.financial_planning import (
    load_inputs, load_and_prepare, make_training_features,
    _fit_and_backtest, recursive_forecast, build_dashboard_payload,
    apply_scenario, SCENARIOS,
)

DATA_DIR = ROOT / 'data' / 'raw'
OUTPUT_JSON = ROOT / 'frontend' / 'public' / 'data' / 'financial_plan.json'
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Project root:', ROOT)
print('Data directory:', DATA_DIR)


## 2. Load the source files

The modelling workflow requires `train.csv`, `test.csv`, `features.csv` and `stores.csv`.

In [ ]:
raw = load_inputs(DATA_DIR)
{key: frame.shape for key, frame in raw.items()}


In [ ]:
for name, frame in raw.items():
    print(f'\n{name.upper()}')
    display(frame.head(3))
    print(frame.dtypes.to_string())


## 3. Prepare and validate the analytical tables

Store-week features are reduced to one record per store/date before joining. This prevents a many-to-many join from inflating sales. Negative weekly sales remain in the historical actuals because the source can contain adjustments; model forecasts are clipped at zero.

In [ ]:
train, future, audit = load_and_prepare(DATA_DIR)
print('Training shape:', train.shape)
print('Forecast schedule shape:', future.shape)
display(pd.json_normalize(audit).T.rename(columns={0: 'value'}))


In [ ]:
display(train[['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday', 'Type', 'Size', 'Temperature', 'Fuel_Price']].head())
print('Historical period:', train.Date.min(), 'to', train.Date.max())
print('Stores:', train.Store.nunique(), '| Departments:', train.Dept.nunique(), '| Weeks:', train.Date.nunique())


## 4. Historical sales performance

Aggregate the source to weekly network sales. The chart is descriptive and shows historical sales, not a forecast.

In [ ]:
weekly_actuals = (train.groupby('Date', as_index=False)
                  .agg(weekly_sales=('Weekly_Sales', 'sum'),
                       holiday=('IsHoliday', 'max'))
                  .sort_values('Date'))
ax = weekly_actuals.plot(x='Date', y='weekly_sales', figsize=(14, 4), legend=False,
                         color='#147b72', linewidth=1.8)
ax.set_title('Walmart sample — total weekly sales')
ax.set_xlabel('Week')
ax.set_ylabel('Sales (USD)')
ax.grid(axis='y', alpha=.2)
plt.show()


## 5. Store format and department mix

In [ ]:
store_type_summary = (train.groupby('Type', as_index=False)
                       .agg(sales=('Weekly_Sales', 'sum'),
                            stores=('Store', 'nunique'),
                            departments=('Dept', 'nunique'),
                            weekly_observations=('Weekly_Sales', 'size'))
                       .sort_values('sales', ascending=False))
display(store_type_summary)


In [ ]:
department_summary = (train.groupby('Dept', as_index=False)
                       .agg(sales=('Weekly_Sales', 'sum'),
                            average_weekly_sales=('Weekly_Sales', 'mean'),
                            stores=('Store', 'nunique'))
                       .sort_values('sales', ascending=False))
display(department_summary.head(15))


## 6. Holiday and external-driver analysis

In [ ]:
holiday_summary = (train.groupby('IsHoliday')['Weekly_Sales']
                   .agg(['count', 'mean', 'median', 'sum'])
                   .rename(index={0: 'Non-holiday', 1: 'Holiday'}))
display(holiday_summary)
if 0 in train.IsHoliday.values and 1 in train.IsHoliday.values:
    h = train.loc[train.IsHoliday == 1, 'Weekly_Sales'].mean()
    nh = train.loc[train.IsHoliday == 0, 'Weekly_Sales'].mean()
    print(f'Average weekly sales difference (holiday vs non-holiday): {(h / nh - 1):.1%}')


In [ ]:
driver_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
               'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
display(train[driver_cols].corr(numeric_only=True)['Weekly_Sales'].sort_values(ascending=False).to_frame('correlation_with_weekly_sales'))


## 7. Feature engineering — strictly backward-looking

The lag and rolling features use `shift(1)` / prior observations only. No current-week target value is allowed into its own feature row.

In [ ]:
X, y, feature_frame = make_training_features(train)
print('Feature matrix:', X.shape)
print('Target:', y.shape)
display(feature_frame[['Store', 'Dept', 'Date', 'Weekly_Sales', 'lag_1', 'lag_4', 'lag_13', 'rolling_mean_4', 'rolling_mean_13', 'history_count']].tail(12))


## 8. Time-based backtest and model fit

The final 13 available historical weeks are held out (or fewer if the dataset has a shorter date range). Forecasts in the holdout are generated recursively: once a week is predicted, that prediction becomes the history for the next week. Holiday rows receive a 5× weight in weighted MAE.

In [ ]:
model, metrics, model_context = _fit_and_backtest(train, future)
display(pd.Series({k: v for k, v in metrics.items() if k != 'validation_weekly'}, name='value').to_frame())


In [ ]:
validation_weekly = pd.DataFrame(metrics['validation_weekly'])
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(validation_weekly['Date'], validation_weekly['actual'], label='Actual', color='#176c69', linewidth=2)
ax.plot(validation_weekly['Date'], validation_weekly['predicted'], label='Recursive prediction', color='#c59b5f', linewidth=2, linestyle='--')
ax.set_title('Recursive holdout: weekly actual vs predicted')
ax.set_ylabel('Sales (USD)'); ax.grid(axis='y', alpha=.2); ax.legend()
plt.xticks(rotation=30); plt.tight_layout(); plt.show()


## 9. Forecast the competition test horizon

The model is trained on the full historical training table. The test file provides future store/department/date rows and the competition feature table supplies known external features where available.

In [ ]:
forecast = recursive_forecast(
    model,
    future,
    train,
    model_context['category_maps'],
    model_context['origin'],
)
forecast = forecast.merge(
    future[['Store', 'Dept', 'Date', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'IsHoliday']],
    on=['Store', 'Dept', 'Date', 'IsHoliday'],
    how='left',
    validate='one_to_one',
)
print('Forecast rows:', len(forecast))
print('Forecast dates:', forecast.Date.min(), 'to', forecast.Date.max())
display(forecast.head())


In [ ]:
weekly_forecast = forecast.groupby('Date', as_index=False).agg(forecast_sales=('forecast_sales', 'sum'))
ax = weekly_forecast.plot(x='Date', y='forecast_sales', figsize=(13, 4), legend=False,
                          color='#147b72', linewidth=2)
ax.set_title('Modelled weekly sales forecast')
ax.set_ylabel('Forecast sales (USD)'); ax.grid(axis='y', alpha=.2)
plt.show()


## 10. Scenario modelling

The statistical model forecasts sales. Planning scenarios then apply explicit demand and price/mix assumptions. The source does not contain actual COGS or operating expenses, so contribution margin and operating expense rates are assumptions rather than observed financials.

In [ ]:
base = apply_scenario(forecast['forecast_sales'], SCENARIOS['Base case'])
upside = apply_scenario(forecast['forecast_sales'], SCENARIOS['Upside'])
conservative = apply_scenario(forecast['forecast_sales'], SCENARIOS['Conservative'])
scenario_summary = pd.DataFrame([
    {'Scenario': name,
     'Revenue': float(values['revenue'].sum()),
     'Modelled gross contribution': float(values['modelled_gross_contribution'].sum()),
     'Modelled operating expenses': float(values['modelled_operating_expenses'].sum()),
     'Modelled operating contribution': float(values['modelled_operating_contribution'].sum())}
    for name, values in [('Base case', base), ('Upside', upside), ('Conservative', conservative)]
])
display(scenario_summary)


In [ ]:
assumptions = SCENARIOS['Base case']
scenario_output = apply_scenario(forecast['forecast_sales'], assumptions)
print('Base case projected revenue:', f"${scenario_output['revenue'].sum():,.0f}")
print('Modelled operating contribution:', f"${scenario_output['modelled_operating_contribution'].sum():,.0f}")


## 11. Build the dashboard data product

The static React/Vite dashboard reads an aggregated JSON file. It does not run Python in the browser and does not provide a data-upload interface.

In [ ]:
payload = build_dashboard_payload(train, forecast, metrics, audit, OUTPUT_JSON)
print('Dashboard JSON:', OUTPUT_JSON)
print('Payload sections:', list(payload.keys()))
print('Forecast horizon:', payload['metadata']['forecast_weeks'], 'weeks')


In [ ]:
pd.DataFrame(payload['forecast_weekly']).head(10)


## 12. Interpretation and limitations

- The historical data covers the competition period only; it is not current Walmart financial performance.
- `Weekly_Sales` is the observed sales measure. The source does not provide COGS, labour, approved budgets or cash-flow statements.
- The seasonal reference plan is an illustrative benchmark constructed from the historical same-ISO-week average by store/department with a 2% planning uplift.
- Contribution and operating contribution are scenario-modelled using editable assumptions; they are not reported profit, EBITDA or cash flow.
- The forecast band uses the P90 absolute weekly percentage error from one time-based backtest. It is an indicative error envelope, not a statistically calibrated confidence interval.
- Markdown fields are sparse in the source and have been filled with zero where absent. Macro indicators are forward-filled by store then median-imputed.
- The model is a portfolio demonstration. A production FP&A process would require approved budgets, finance-owned cost drivers, reconciliation, forecast governance and additional rolling-origin validation.

**Reproducibility:** run `python run_analysis.py` from the repository root to regenerate the same dashboard data product consumed by the frontend.